In [2]:
import os, re, sqlite3
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

In [3]:
#Prints all the supported models for use
from google import genai
client = genai.Client()
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

response = llm.invoke("Explain what a vector embedding is in two sentences.")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'A vector embedding is a numerical representation of complex data, such as words, images, or audio, translated into an array of numbers that computers can process. It captures the underlying meaning and context of that data by positioning similar concepts closer together within a multi-dimensional mathematical space.', 'extras': {'signature': 'EscQCsQQAWkUfROCwKplEzSV2V932qOzYZMVmizWWn5UdEWtlBKwwhn7gZgKuU9hYkQh4mGvPbp0ujNTE1bjpk0wuPrmqJNLqFmkCnuLxFtUVPgqmWEItJiJyKqherkUl/c7Miwe0AVIf+IjBMQnHqmnG3lno+mv+aj7s9MKsyFEnf1Ovkz24q1Gvk8Gt7QgGvteXldGlPeFS/Njt6PNBI/rhoZJZpNFR5PwByJcITn03RdWpJLs14yRR7wMKqbdGHSskUJ+wYy8pMvH3TgDKBFmoK0qLXphVqO0ywg82JSgyqaErQJK4o+iwbsBQPKsVkjbe+P4szbk/5K1qJVNFu467Yg8lsUsX6C+vwhOngrDBNqbQiV+3poWNAcMYqvoy8NtbwQ7kRXG2l2TY1oWY8NbebQ8tOWfBwsWG4b2q5m81lCpX7h6GhOZ9ypsQTXVnSp2pClqrW31I5a5oJjBa5S5K5j1gDQh1dWFBfNwdcY9SOvm06uadHleeip4nJAZuXX71BG2Uhaj+XSpw+6wlh58HsiJE/ghoSBwrPDsLXkH5PSmmjLJyKkGhXiOwF9CSyJBfWwuUdYQNlwBTeRS+sEzfBm6lv7n2OVeDFDc9LnZDpiig9/v